# **Univariate Forecasting**

https://huggingface.co/amazon/chronos-2

In [ ]:
# !pip install git+https://github.com/amazon-science/chronos-forecasting.git
import pandas as pd  # requires: pip install 'pandas[pyarrow]'
import numpy as np
from chronos import Chronos2Pipeline

# 1. Load the official Chronos-2 pipeline
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda" if torch.cuda.is_available() else "cpu")

# 2. Load the intermediate processed data
context_df = pd.read_csv("context_df.csv")
ground_truth = np.load("horizon_true.npy")
# print(context_df)
# print(ground_truth)

# Ensure the timestamp column is parsed as datetime objects for the API
context_df["timestamp"] = pd.to_datetime(context_df["timestamp"])

# 3. Generate predictions
pred_df = pipeline.predict_df(
    context_df,
    prediction_length=24,  # Number of steps (hours) to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast
    id_column="id",  # Column identifying different time series
    timestamp_column="timestamp",  # Column with datetime information
    target="target",  # Column(s) with time series values to predict
)

print(pred_df)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

     id           timestamp target_name  predictions          0.1  \
0   NSW 2015-02-09 01:00:00      target  4633.734375  4500.312988   
1   NSW 2015-02-09 02:00:00      target  4429.503906  4238.555176   
2   NSW 2015-02-09 03:00:00      target  4456.013672  4186.129883   
3   NSW 2015-02-09 04:00:00      target  4832.680176  4501.832520   
4   NSW 2015-02-09 05:00:00      target  5686.690430  5224.156250   
5   NSW 2015-02-09 06:00:00      target  6610.991211  5988.332031   
6   NSW 2015-02-09 07:00:00      target  7038.325195  6327.121094   
7   NSW 2015-02-09 08:00:00      target  7282.292480  6550.623535   
8   NSW 2015-02-09 09:00:00      target  7417.970215  6657.963867   
9   NSW 2015-02-09 10:00:00      target  7427.785645  6714.414551   
10  NSW 2015-02-09 11:00:00      target  7479.526367  6730.042480   
11  NSW 2015-02-09 12:00:00      target  7586.797852  6770.525879   
12  NSW 2015-02-09 13:00:00      target  7669.201660  6718.468262   
13  NSW 2015-02-09 14:00:00      t

In [ ]:
# 4. Extract the Median Predictions
forecast_median = pred_df["0.5"].values # Or equivalently: pred_df["predictions"].values

# 5. Define Evaluation Metrics
def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    # Root Mean Squared Error
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # Mean Absolute Percentage Error
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return rmse, mape

# 6. Run Evaluation
rmse_score, mape_score = calculate_metrics(ground_truth, forecast_median)

print("--- Chronos-2 Univariate Baseline Metrics ---")
print(f"Ground Truth Shape: {ground_truth.shape}")  # Outputs: (24,)
print(f"Forecast Shape:     {forecast_median.shape}")  # Outputs: (24,)
print(f"RMSE: {rmse_score:.4f} MW")
print(f"MAPE: {mape_score:.4f}%")

--- Chronos-2 Univariate Baseline Metrics ---
Ground Truth Shape: (24,)
Forecast Shape:     (24,)
RMSE: 171.6731 MW
MAPE: 2.2609%
